In [ ]:
from pathlib import Path
from frameit.core.settings_class import SimulationConfig
from frameit.core.runner import frameitRunner
from frameit.processing.extraction import center2box

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import math


conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_batsirai.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_belna.yml"))

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_chido.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_ianos.yml"))

conf.printID()

runner = frameitRunner(conf)
runner.run()

In [ ]:
runner.track

### CAS MNH

In [ ]:

import matplotlib.pyplot as plt
import cartopy.crs as ccrs

grid_ds = runner.ds_user['level']
track_ds = runner.track

lon2d = grid_ds["longitude"].values
lat2d = grid_ds["latitude"].values

lon_track = grid_ds["longitude"][0,:].values[track_ds["cx"].values]   # ici cx = longitude
lat_track = grid_ds["latitude"][:,0].values[track_ds["cy"].values]   # ici cy = latitude

# print(lat_track,lon_track)

# bornes de la traj + petite marge
margin = 2.0  # en degrés
lon_min = lon_track.min() - margin
lon_max = lon_track.max() + margin
lat_min = lat_track.min() - margin
lat_max = lat_track.max() + margin

ncols = 2
nt = track_ds.sizes["time"]
nrows = math.ceil(nt / ncols)
fig, axes = plt.subplots(figsize=(5 * ncols, 4 * nrows),ncols=ncols,nrows=nrows,subplot_kw={"projection": ccrs.PlateCarree()})
axes = np.atleast_1d(axes).ravel()
    
for it in range(nt):
        
    ax = axes[it]
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.gridlines(draw_labels=True, linewidth=0.5, linestyle=":")

    # fond de carte
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # trajectoire complète
    ax.plot(
        lon_track, lat_track, "-k",
        transform=ccrs.PlateCarree()
    )

    # centre au temps courant
    ax.plot(
        lon_track[it], lat_track[it], "or",
        transform=ccrs.PlateCarree()
    )

    # indices de la boîte domainpée
    ixmin = int(track_ds["ix_min_domain"].values[it])
    ixmax = int(track_ds["ix_max_domain"].values[it])
    iymin = int(track_ds["iy_min_domain"].values[it])
    iymax = int(track_ds["iy_max_domain"].values[it])

    # coins de la boîte (ordre fermé)
    xs = [ixmin, ixmax, ixmax, ixmin, ixmin]
    ys = [iymin, iymin, iymax, iymax, iymin]
    box_lon = lon2d[ys, xs]
    box_lat = lat2d[ys, xs]

    # tracé de la boîte en lon/lat
    ax.plot(
        box_lon, box_lat, "-r",
        transform=ccrs.PlateCarree()
    )
    ax.set_title(f'{track_ds["time"][it].values}')
    ax.set_xlabel(grid_ds["longitude"][0,:].values)

plt.tight_layout()


# masquer les panneaux inutilisés
for k in range(nt, nrows * ncols):
    fig.delaxes(axes[k])

### CAS AROME

In [ ]:
import math
import numpy as np
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

track_ds = runner.track
grid_ds = runner.ds_user['heightAboveGround']

lon1d = grid_ds["longitude"].values
lat1d = grid_ds["latitude"].values

cx = track_ds["cx"].values.astype(int)
cy = track_ds["cy"].values.astype(int)

lon_track = lon1d[cx]
lat_track = lat1d[cy]

# bornes de la traj + petite marge
margin = 5.0
lon_min = lon_track.min() - margin
lon_max = lon_track.max() + margin
lat_min = lat_track.min() - margin
lat_max = lat_track.max() + margin

# tailles de la boîte en km (stockées par center2box)
box_x_km = track_ds.attrs["box_x_size_km"]
box_y_km = track_ds.attrs["box_y_size_km"]

ncols = 2
nt = track_ds.sizes["time"]
nrows = math.ceil(nt / ncols)
fig, axes = plt.subplots(
    figsize=(5 * ncols, 4 * nrows),
    ncols=ncols,
    nrows=nrows,
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = np.atleast_1d(axes).ravel()

for it in range(nt):
    ax = axes[it]
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.gridlines(draw_labels=True, linewidth=0.5, linestyle=":")

    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # trajectoire complète
    ax.plot(
        lon_track, lat_track, "-k",
        transform=ccrs.PlateCarree(),
    )

    # centre au temps courant
    lon_c = lon_track[it]
    lat_c = lat_track[it]
    ax.plot(
        lon_c, lat_c, "or",
        transform=ccrs.PlateCarree(),
    )

    # conversion taille km -> degrés autour du centre
    # ~111 km par degré de latitude
    dy_deg = box_y_km / 111.0
    # ~111 km * cos(lat) par degré de longitude
    dx_deg = box_x_km / (111.0 * np.cos(np.deg2rad(lat_c)))

    lon_left = lon_c - dx_deg / 2.0
    lat_bottom = lat_c - dy_deg / 2.0
    width = dx_deg
    height = dy_deg

    # boîte idéale : toujours centrée, taille constante, peut dépasser le domaine
    ax.add_patch(
        Rectangle(
            (lon_left, lat_bottom),
            width,
            height,
            ls="--",
            edgecolor="red",
            facecolor="none",
            lw=1.5,
            transform=ccrs.PlateCarree(),
        )
    )

    # tracé des bords du domaine (optionnel)
    ax.plot(
        [lon1d.min(), lon1d.min(), lon1d.max(), lon1d.max(), lon1d.min()],
        [lat1d.min(), lat1d.max(), lat1d.max(), lat1d.min(), lat1d.min()],
        "--k",
        lw=2,
        transform=ccrs.PlateCarree(),
    )

    ax.set_title(str(track_ds["time"][it].values))

plt.tight_layout()

for k in range(nt, nrows * ncols):
    fig.delaxes(axes[k])